<div class="lesson-banner">
<span class="lesson-kicker">Python course · 2-hour lesson</span>
<p>Use DB-API connections, parameterized SQL, transactions, row factories, and repository boundaries safely.</p>
</div>

## Learning objectives

- Connect to SQLite with context-managed transactions.
- Use parameterized queries instead of string-built SQL.
- Understand commit, rollback, and transaction atomicity.
- Separate persistence from domain transformation.

::: {.callout-note}
### How to use this notebook
Read the explanation, predict each result, run the code, change the inputs, and complete the practice before revealing the solution.
:::


## DB-API and transactions

Python database drivers follow a common pattern: connect, create a cursor, execute, fetch, and close. A transaction groups changes into one atomic unit. With SQLite, the connection context commits on success and rolls back when an exception escapes the block.


In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row

with connection:
    connection.execute('''
        CREATE TABLE course (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL UNIQUE,
            hours INTEGER NOT NULL CHECK (hours > 0)
        )
    ''')
    connection.executemany(
        "INSERT INTO course(name, hours) VALUES (?, ?)",
        [("Python", 45), ("SQL", 30), ("Machine Learning", 60)],
    )

rows = connection.execute("SELECT name, hours FROM course ORDER BY hours").fetchall()
print([dict(row) for row in rows])


## Parameters prevent SQL injection

Never interpolate external values into SQL text. Placeholders let the driver transmit values separately from the SQL program and handle quoting correctly. Column names and sort direction cannot normally be parameterized; map those choices through a strict allow-list.


In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE learner(name TEXT, score REAL)")
connection.executemany(
    "INSERT INTO learner VALUES (?, ?)",
    [("Asha", 88), ("Ravi", 72), ("Meera", 91)],
)

minimum = 80
rows = connection.execute(
    "SELECT name, score FROM learner WHERE score >= ? ORDER BY score DESC",
    (minimum,),
).fetchall()
print(rows)


## Repository boundaries

A repository can isolate SQL from domain logic, but it should not hide transaction decisions or return vague untyped data everywhere. Keep queries focused, select only needed columns, enforce constraints in the database, and test persistence behavior against a temporary database.


In [ ]:
class CourseRepository:
    def __init__(self, connection):
        self.connection = connection

    def find_by_minimum_hours(self, minimum: int) -> list[dict]:
        rows = self.connection.execute(
            "SELECT name, hours FROM course WHERE hours >= ? ORDER BY hours DESC",
            (minimum,),
        ).fetchall()
        return [dict(row) for row in rows]


# repository = CourseRepository(connection)
# print(repository.find_by_minimum_hours(40))


## Worked example: atomic enrollment

The unique constraint prevents duplicates and the transaction keeps the operation all-or-nothing.


In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute('''
    CREATE TABLE enrollment (
        learner TEXT NOT NULL,
        course TEXT NOT NULL,
        enrolled_at TEXT NOT NULL,
        UNIQUE (learner, course)
    )
''')


def enroll(connection, learner: str, course: str, enrolled_at: str) -> None:
    with connection:
        connection.execute(
            "INSERT INTO enrollment VALUES (?, ?, ?)",
            (learner, course, enrolled_at),
        )


enroll(connection, "Asha", "Python", "2026-09-09")
print(connection.execute("SELECT * FROM enrollment").fetchall())


## Practice lab

Complete these tasks without copying the solution. Test normal, boundary, and invalid inputs where relevant.

1. Create course and enrollment tables with primary, foreign, unique, and check constraints.
2. Insert three records using `executemany`.
3. Query with a parameterized minimum score.
4. Force an error halfway through a transaction and verify that earlier work was rolled back.

::: {.callout-important}
### Practice standard
Your answer should be readable, deterministic, and divided into small functions when the task contains more than one rule.
:::


## Suggested solution

Open the folded code only after attempting every task.


In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute("PRAGMA foreign_keys = ON")
connection.executescript('''
    CREATE TABLE course (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL UNIQUE
    );
    CREATE TABLE enrollment (
        learner TEXT NOT NULL,
        course_id INTEGER NOT NULL REFERENCES course(id),
        score REAL CHECK (score BETWEEN 0 AND 100),
        UNIQUE (learner, course_id)
    );
''')

with connection:
    connection.executemany("INSERT INTO course(name) VALUES (?)", [("Python",), ("SQL",)])
    connection.executemany(
        "INSERT INTO enrollment VALUES (?, ?, ?)",
        [("Asha", 1, 88), ("Ravi", 1, 72), ("Asha", 2, 91)],
    )

minimum = 80
rows = connection.execute(
    "SELECT learner, score FROM enrollment WHERE score >= ? ORDER BY score DESC",
    (minimum,),
).fetchall()
print(rows)


## Knowledge check

**1. Why parameterize values?**

::: {.callout-note collapse="true"}
### Answer
It separates data from SQL syntax and prevents injection.
:::

**2. What does rollback provide?**

::: {.callout-note collapse="true"}
### Answer
Failed transaction changes are not partially persisted.
:::

**3. Where should core data rules live?**

::: {.callout-note collapse="true"}
### Answer
At appropriate layers, including database constraints for persisted invariants.
:::


## Recap

- Parameterize every external value.
- Make transaction boundaries explicit.
- Use constraints and focused repository methods.


<div class="lesson-nav">
<a href="15-apis-and-automation.html"><i class="bi bi-arrow-left" aria-hidden="true"></i> APIs, HTTP, and Automation</a>
<a href="17-concurrency-and-performance.html">Concurrency, Performance, and Memory <i class="bi bi-arrow-right" aria-hidden="true"></i></a>
</div>
